In [9]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
dataset_dir = '/content/drive/MyDrive/rgb'

files = sorted(os.listdir(dataset_dir))
print(f"Total images: {len(files)}")
print(files[:3])

Total images: 613
['1305031452.791720.png', '1305031452.823674.png', '1305031452.859642.png']


In [11]:
def run_sift_match(path1, path2):
    img1_gray = cv2.imread(path1, cv2.IMREAD_GRAYSCALE)
    img2_gray = cv2.imread(path2, cv2.IMREAD_GRAYSCALE)
    img1_rgb = cv2.cvtColor(cv2.imread(path1), cv2.COLOR_BGR2RGB)
    img2_rgb = cv2.cvtColor(cv2.imread(path2), cv2.COLOR_BGR2RGB)

    sift = cv2.SIFT_create()
    kp1, desc1 = sift.detectAndCompute(img1_gray, None)
    kp2, desc2 = sift.detectAndCompute(img2_gray, None)

    bf = cv2.BFMatcher()
    matches = bf.knnMatch(desc1, desc2, k=2)

    good_matches = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            good_matches.append(m)

    pts1 = np.float32([kp1[m.queryIdx].pt for m in good_matches])
    pts2 = np.float32([kp2[m.trainIdx].pt for m in good_matches])

    num_matches = len(good_matches)

    if num_matches >= 8:
        F, mask = cv2.findFundamentalMat(pts1, pts2, cv2.FM_RANSAC, ransacReprojThreshold=1.0, confidence=0.99)
        num_inliers = int(mask.sum()) if mask is not None else 0
    else:
        mask = None
        num_inliers = 0

    inlier_ratio = num_inliers / num_matches if num_matches > 0 else 0

    return pts1, pts2, mask, img1_rgb, img2_rgb, num_matches, num_inliers, inlier_ratio


def visualize_sift_matches(path1, path2, label, max_draw=100):
    pts1, pts2, mask, img1_rgb, img2_rgb, num_matches, num_inliers, inlier_ratio = run_sift_match(path1, path2)

    print(f"--- {label} ---")
    print(f"Matches: {num_matches} | Inliers: {num_inliers} | Inlier ratio: {inlier_ratio:.2%}")

    h1, w1 = img1_rgb.shape[:2]
    h2, w2 = img2_rgb.shape[:2]
    canvas = np.zeros((max(h1, h2), w1 + w2, 3), dtype=np.uint8)
    canvas[:h1, :w1] = img1_rgb
    canvas[:h2, w1:w1+w2] = img2_rgb

    if mask is not None:
        inlier_idx = np.where(mask.ravel() == 1)[0]
        draw_idx = inlier_idx[:max_draw]

        plt.figure(figsize=(16, 8))
        plt.imshow(canvas)
        for idx in draw_idx:
            x1, y1 = pts1[idx]
            x2, y2 = pts2[idx]
            plt.plot([x1, x2 + w1], [y1, y2], 'g-', linewidth=0.5)
            plt.plot(x1, y1, 'ro', markersize=2)
            plt.plot(x2 + w1, y2, 'ro', markersize=2)
        plt.title(f"SIFT: {label} — {len(draw_idx)} inliers shown (of {num_inliers} total)")
        plt.axis('off')
        plt.show()

    return {'test': label, 'matches': num_matches, 'inliers': num_inliers, 'inlier_ratio': inlier_ratio}

In [15]:
sift_robustness_results = []
sift_robustness_results.append(visualize_sift_matches(pt_path, t45_path, "trolley normal vs rotated"))
sift_robustness_results.append(visualize_sift_matches(pt_path, zt_path, "trolley normal vs zoomed"))
sift_robustness_results.append(visualize_sift_matches(pw_path, r90_path, "wall normal vs rotated"))
sift_robustness_results.append(visualize_sift_matches(pw_path, zw_path, "wall normal vs zoomed"))

Output hidden; open in https://colab.research.google.com to view.

In [16]:
import pandas as pd
sift_robustness_df = pd.DataFrame(sift_robustness_results)
sift_robustness_df.to_csv('/content/drive/MyDrive/sift_robustness_tests.csv', index=False)
print(sift_robustness_df)

                        test  matches  inliers  inlier_ratio
0  trolley normal vs rotated     2981     2068      0.693727
1   trolley normal vs zoomed     1612      937      0.581266
2     wall normal vs rotated       22        9      0.409091
3      wall normal vs zoomed      105       62      0.590476
